# Phase 5.1 — Real Evaluation Validation (Google Colab T4)

**Purpose**: Validate the Evaluation Service end-to-end on a REAL LoRA adapter
produced in Phase 4.3, against a small validation dataset.

**Pipeline**: `Phase 4.3 adapter → load base + LoRA → generate predictions → compute metrics → report`

**Metrics**: ROUGE-L, BERTScore (P/R/F1), Semantic Similarity

---

**Prerequisites**:
1. Phase 4.3 notebook has been run and `qlora_phase43_results/adapter/` exists in Google Drive.
2. Runtime type is **T4 GPU**.
3. HuggingFace token with Gemma access approved.


## Step 1: Install Dependencies

Install the evaluation metric libraries (not needed for training, only for evaluation).
Also install `peft` (not in the Colab default set) for adapter loading.

In [1]:
# Step 1: Install evaluation dependencies
# peft + transformers already installed in Colab; ensure metric libs are present
!pip install -q peft>=0.13.0 rouge-score>=0.1.2 bert-score>=0.3.13 sentence-transformers>=2.2.2

## Step 2: Verify GPU + Library Versions

In [2]:
# Step 2: Verify GPU + library versions
import torch, transformers, sys
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('CUDA:', torch.cuda.is_available())
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
try:
    import peft; print('peft:', peft.__version__)
except ImportError:
    print('peft: NOT INSTALLED — re-run Step 1')
try:
    import rouge_score; print('rouge_score: OK')
except ImportError:
    print('rouge_score: NOT INSTALLED — re-run Step 1')
try:
    import bert_score; print('bert_score: OK')
except ImportError:
    print('bert_score: NOT INSTALLED — re-run Step 1')
try:
    import sentence_transformers; print('sentence_transformers: OK')
except ImportError:
    print('sentence_transformers: NOT INSTALLED — re-run Step 1')
assert torch.cuda.is_available(), 'Need a GPU runtime'


GPU: Tesla T4
CUDA: True
torch: 2.11.0+cu128
transformers: 5.12.0
peft: 0.19.1
rouge_score: OK
bert_score: OK
sentence_transformers: OK


## Step 3: Authenticate with HuggingFace

Required to download the Gemma 3 1B IT base model (the adapter's base).

In [5]:
# Step 3: HuggingFace Authentication
from huggingface_hub import login
import os

# Option A: Use Colab secrets (recommended)
# Add your HF token as a Colab secret named "HF_TOKEN"
# (Click the key icon 🔑 in the left sidebar)
hf_token = os.environ.get("HF_TOKEN") or None

# Option B: Paste token directly (less secure, for quick testing)
# hf_token = "hf_YOUR_TOKEN_HERE"

if hf_token:
    login(token=hf_token, add_to_git_credential=True)
    print("✅ HuggingFace authentication successful")
else:
    print("⚠️  No HF_TOKEN found. You will be prompted interactively.")
    print("   Alternatively, set HF_TOKEN in Colab secrets (key icon 🔑)")
    login(add_to_git_credential=True)  # Interactive prompt


⚠️  No HF_TOKEN found. You will be prompted interactively.
   Alternatively, set HF_TOKEN in Colab secrets (key icon 🔑)


## Step 4: Mount Google Drive + Locate Phase 4.3 Adapter

The Phase 4.3 notebook saved the adapter to `MyDrive/qlora_phase43_results/adapter/`.

In [6]:
# Step 4: Mount Drive + locate adapter
from google.colab import drive
import os, json
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/qlora_phase43_results'
ADAPTER_DIR = os.path.join(DRIVE_DIR, 'adapter')
TOKENIZER_DIR = os.path.join(ADAPTER_DIR, 'tokenizer')
print('Adapter dir:', ADAPTER_DIR)
print('Exists:', os.path.isdir(ADAPTER_DIR))
print('Tokenizer dir:', TOKENIZER_DIR)
print('Tokenizer exists:', os.path.isdir(TOKENIZER_DIR))
print('\nAdapter contents:')
for f in sorted(os.listdir(ADAPTER_DIR)):
    p = os.path.join(ADAPTER_DIR, f)
    sz = os.path.getsize(p) if os.path.isfile(p) else 0
    print(f'  {f} ({sz/1024:.1f} KB)' if os.path.isfile(f'ADAPTER_DIR/{f}') else f'  {f}/')
print('\nTokenizer contents:')
for f in sorted(os.listdir(TOKENIZER_DIR)):
    print(f'  {f}')
assert os.path.isdir(ADAPTER_DIR), 'Adapter dir not found — run Phase 4.3 first'
assert os.path.isdir(TOKENIZER_DIR), 'Tokenizer dir not found inside adapter'
# Read training metadata to get base_model
meta_path = os.path.join(ADAPTER_DIR, 'training_metadata.json')
with open(meta_path) as f:
    training_meta = json.load(f)
BASE_MODEL = training_meta['base_model']
print(f'\nBase model from metadata: {BASE_MODEL}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter dir: /content/drive/MyDrive/qlora_phase43_results/adapter
Exists: True
Tokenizer dir: /content/drive/MyDrive/qlora_phase43_results/adapter/tokenizer
Tokenizer exists: True

Adapter contents:
  README.md/
  adapter_config.json/
  adapter_model.safetensors/
  tokenizer/
  training_metadata.json/

Tokenizer contents:
  chat_template.jinja
  tokenizer.json
  tokenizer_config.json

Base model from metadata: google/gemma-3-1b-it


## Step 5: Validate Adapter Integrity

Confirm `PeftModel.from_pretrained()` loads successfully and the tokenizer loads.
No mocks — real model loading.

In [7]:
# Step 5: Validate adapter + tokenizer load
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import time

print('Loading tokenizer from:', TOKENIZER_DIR)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer loaded OK. vocab_size:', tokenizer.vocab_size)

print('\nLoading base model with 4-bit NF4 quantization (matches Phase 4.3)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    attn_implementation='eager',
    device_map='auto',
)
print(f'Base model loaded in {time.time()-t0:.1f}s')

print('\nLoading LoRA adapter from:', ADAPTER_DIR)
t0 = time.time()
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
print(f'Adapter loaded in {time.time()-t0:.1f}s')
model.eval()

vram = torch.cuda.memory_allocated() / (1024**3)
print(f'\nVRAM after load: {vram:.2f} GB')
print('Adapter config:', model.peft_config)
print('\nALL LOADS OK — adapter + tokenizer + base model verified')


Loading tokenizer from: /content/drive/MyDrive/qlora_phase43_results/adapter/tokenizer


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Tokenizer loaded OK. vocab_size: 262144

Loading base model with 4-bit NF4 quantization (matches Phase 4.3)...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Base model loaded in 7.1s

Loading LoRA adapter from: /content/drive/MyDrive/qlora_phase43_results/adapter
Adapter loaded in 0.5s

VRAM after load: 0.95 GB
Adapter config: {'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path='google/gemma-3-1b-it', revision=None, inference_mode=True, r=16, target_modules={'up_proj', 'gate_proj', 'q_proj', 'down_proj', 'k_proj', 'o_proj', 'v_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=Lo

## Step 6: Quick Inference Smoke Test

Generate one prediction to verify the full inference path works before running the eval dataset.

In [8]:
# Step 6: Inference smoke test
test_prompt = '### Instruction:\nWhat is 2+2?\n\n### Response:\n'
inputs = tokenizer(test_prompt, return_tensors='pt')
inputs = {k: v.to(model.device) for k, v in inputs.items()}
input_len = inputs['input_ids'].shape[-1]
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
prediction = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
print('PROMPT:', repr(test_prompt))
print('PREDICTION:', repr(prediction))
print('\nInference smoke test OK')


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


PROMPT: '### Instruction:\nWhat is 2+2?\n\n### Response:\n'
PREDICTION: '2 + 2 = 4\n\n### Answer:\n4\n'

Inference smoke test OK


## Step 7: Load Validation Dataset

Load 20 examples from `yahma/alpaca-cleaned` (disjoint from the 50 used in training —
we take examples 100-119 to avoid overlap with the first 50).

In [9]:
# Step 7: Load validation dataset
from datasets import load_dataset
NUM_EVAL_SAMPLES = 20
ds = load_dataset('yahma/alpaca-cleaned', split='train')
# Use examples 100-119 (training used 0-49)
ds_eval = ds.select(range(100, 100 + NUM_EVAL_SAMPLES))
print(f'Loaded {len(ds_eval)} eval examples (indices 100-{100+NUM_EVAL_SAMPLES-1})')

def format_alpaca(example):
    instruction = example.get('instruction', '')
    input_text = example.get('input', '')
    output_text = example.get('output', '')
    if input_text and input_text.strip():
        prompt = f'### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n'
    else:
        prompt = f'### Instruction:\n{instruction}\n\n### Response:\n'
    return {'prompt': prompt, 'reference': output_text}

ds_formatted = ds_eval.map(format_alpaca)
print('Sample prompt:', ds_formatted[0]['prompt'][:120])
print('Sample reference:', ds_formatted[0]['reference'][:120])


Loaded 20 eval examples (indices 100-119)


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Sample prompt: ### Instruction:
Use the following pieces of context to answer the question at the end. If you don't know the answer, ju
Sample reference: I'm sorry, but I don't have enough contextual information about the Epson F7100 to answer that question.


## Step 8: Generate Predictions

Run the model on all eval prompts. Greedy decoding (do_sample=False) for reproducibility.

In [10]:
# Step 8: Generate predictions
import time
print(f'Generating predictions for {len(ds_formatted)} examples...')
predictions = []
t0 = time.time()
for i, ex in enumerate(ds_formatted):
    inputs = tokenizer(ex['prompt'], return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs['input_ids'].shape[-1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    pred = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
    predictions.append(pred)
    if (i + 1) % 5 == 0:
        print(f'  {i+1}/{len(ds_formatted)} done')
gen_time = time.time() - t0
print(f'\nGenerated {len(predictions)} predictions in {gen_time:.1f}s')
print('\n--- Sample Predictions ---')
for i in range(3):
    print(f'\n[{i+1}] REF: {ds_formatted[i]["reference"][:100]}')
    print(f'    PRED: {predictions[i][:100]}')


Generating predictions for 20 examples...
  5/20 done
  10/20 done
  15/20 done
  20/20 done

Generated 20 predictions in 285.5s

--- Sample Predictions ---

[1] REF: I'm sorry, but I don't have enough contextual information about the Epson F7100 to answer that quest
    PRED: Yes.


[2] REF: Kittens - noun
often - adverb
scamper - verb
around - preposition
excitedly - adverb.
    PRED: *   Kittens: Noun
*   often: Adverb
*   scamper: Verb
*   around: Adverb
*   excitedly: Adverb

### 

[3] REF: Here is a randomly generated 8-character password: rT8$jLpZ.

Please note that it is advisable to us
    PRED: ```
12345678
```
### Explanation:
The password is generated by using the characters 1, 2, 3, 4, 5, 6


## Step 9: Compute Metrics

Compute the three MVP metrics: ROUGE-L, BERTScore, Semantic Similarity.
These are the same functions used in `backend/app/services/metrics.py`.

In [11]:
# Step 9: Compute metrics
references = [ex['reference'] for ex in ds_formatted]
print('Computing ROUGE-L...')
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = [scorer.score(ref, pred)['rougeL'].fmeasure
                for ref, pred in zip(references, predictions)]
rouge_l = sum(rouge_scores) / len(rouge_scores)
print(f'  ROUGE-L F1: {rouge_l:.4f}')

print('\nComputing BERTScore (downloads a model on first run)...')
from bert_score import score as bert_score_fn
P, R, F = bert_score_fn(predictions, references, lang='en', verbose=False)
bertscore_p = float(P.mean().item())
bertscore_r = float(R.mean().item())
bertscore_f1 = float(F.mean().item())
print(f'  BERTScore P: {bertscore_p:.4f}')
print(f'  BERTScore R: {bertscore_r:.4f}')
print(f'  BERTScore F1: {bertscore_f1:.4f}')

print('\nComputing Semantic Similarity...')
from sentence_transformers import SentenceTransformer, util
st_model = SentenceTransformer('all-MiniLM-L6-v2')
pred_emb = st_model.encode(predictions, convert_to_tensor=True)
ref_emb = st_model.encode(references, convert_to_tensor=True)
cos = util.cos_sim(pred_emb, ref_emb)
diag = cos.diagonal()
sem_sim = float(((diag + 1.0) / 2.0).mean().item())
print(f'  Semantic Similarity: {sem_sim:.4f}')

print('\n' + '='*60)
print('METRICS SUMMARY')
print('='*60)
print(f'  ROUGE-L:              {rouge_l:.4f}')
print(f'  BERTScore Precision:  {bertscore_p:.4f}')
print(f'  BERTScore Recall:     {bertscore_r:.4f}')
print(f'  BERTScore F1:         {bertscore_f1:.4f}')
print(f'  Semantic Similarity:  {sem_sim:.4f}')
print('='*60)


Computing ROUGE-L...
  ROUGE-L F1: 0.2338

Computing BERTScore (downloads a model on first run)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Semantic Similarity: 0.7704

METRICS SUMMARY
  ROUGE-L:              0.2338
  BERTScore Precision:  0.8191
  BERTScore Recall:     0.8709
  BERTScore F1:         0.8425
  Semantic Similarity:  0.7704


## Step 10: Save Evaluation Report to Drive

Persist the evaluation results so they survive Colab session termination.

In [ ]:
# Step 10: Save evaluation report
import json, platform
from datetime import datetime
EVAL_REPORT = {
    'phase': '5.1',
    'timestamp': datetime.now().isoformat(),
    'environment': {
        'gpu_name': torch.cuda.get_device_name(0),
        'gpu_vram_gb': round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
        'torch_version': torch.__version__,
        'transformers_version': transformers.__version__,
        'python_version': platform.python_version(),
    },
    'adapter': {
        'base_model': BASE_MODEL,
        'adapter_path': ADAPTER_DIR,
        'training_metadata': training_meta,
    },
    'dataset': {
        'name': 'yahma/alpaca-cleaned',
        'eval_split': 'indices 100-119',
        'num_samples': len(ds_formatted),
    },
    'metrics': {
        'rouge_l': round(rouge_l, 4),
        'bertscore_precision': round(bertscore_p, 4),
        'bertscore_recall': round(bertscore_r, 4),
        'bertscore_f1': round(bertscore_f1, 4),
        'semantic_similarity': round(sem_sim, 4),
    },
    'prediction_time_s': round(gen_time, 2),
    'predictions': [{'prompt': ex['prompt'][:200], 'reference': ex['reference'][:200], 'prediction': pred[:200]} for ex, pred in zip(ds_formatted, predictions)],
}
report_path = os.path.join(DRIVE_DIR, 'phase51_evaluation_report.json')
with open(report_path, 'w') as f:
    json.dump(EVAL_REPORT, f, indent=2)
print(f'Evaluation report saved to: {report_path}')
print('\nValidation complete. Paste the METRICS SUMMARY output into docs/24.')


## Step 11: Verification Checklist

After running all steps, verify:
1. PeftModel.from_pretrained() loaded successfully (Step 5)
2. Tokenizer loaded from `adapter/tokenizer/` (Step 5)
3. Inference produced non-empty predictions (Step 6 + Step 8)
4. ROUGE-L computed (Step 9)
5. BERTScore computed (Step 9)
6. Semantic Similarity computed (Step 9)
7. Report saved to Drive (Step 10)

If all 7 pass, the Evaluation Service has been validated end-to-end on real artifacts.